
# Titanic Data Preprocessing Exercises

🌟 Exercise 1: Duplicate Detection and Removal
Instructions
Objective: Identify and remove duplicate entries in the Titanic dataset.

Load the Titanic dataset.
Identify if there are any duplicate rows based on all columns.
Remove any duplicate rows found in the dataset.
Verify the removal of duplicates by checking the number of rows before and after the duplicate removal.
Hint: Use the duplicated() and drop_duplicates() functions in Pandas.



🌟 Exercise 2: Handling Missing Values
Instructions
Identify columns in the Titanic dataset with missing values.
Explore different strategies for handling missing data, such as removal, imputation, and filling with a constant value.
Apply each strategy to different columns based on the nature of the data.
Hint: Review methods like dropna(), fillna(), and SimpleImputer from scikit-learn.



🌟 Exercise 3: Feature Engineering
Instructions
Create new features, such as Family Size from SibSp and Parch, and Title extracted from the Name column.
Convert categorical variables into numerical form using techniques like one-hot encoding or label encoding.
You will encode new categorical features (like Title) here, but do not scale numerical features yet — that will come after outlier handling.
Hint: Utilize Pandas for data manipulation and scikit-learn’s preprocessing module for encoding.



🌟 Exercise 4: Outlier Detection and Handling
Goal: Detect and cap or transform outliers in columns like Fare and Age.

1. Visualize distributions using boxplots or histograms to identify potential outliers.
2. Use IQR or Z-score methods to detect them.
3. Handle outliers with:

Quantile capping (e.g. 0.98)
Log transformation
Row removal
4. Compare the dataset before and after treatment.

📌 Note: Small differences between 0.98 and 0.99 quantiles are normal when extreme values are rare or far apart. Use df.quantile() to explore and choose thresholds empirically, backed by visualization.



🌟 Exercise 5: Data Standardization and Normalization
Goal: Scale numerical features to prepare for modeling.

Use StandardScaler (mean = 0, std = 1) for normally distributed features.
Use MinMaxScaler (range [0, 1]) for features that are skewed or bounded.
📌 Important: Perform this step after outlier treatment to avoid distortion caused by extreme values.



🌟 Exercise 6: Feature Encoding
Goal: Finalize categorical variable encoding.

1. Identify remaining categorical columns (e.g. Sex, Embarked, Title).
2. Apply:

One-Hot Encoding for nominal variables.
Label Encoding if any ordinal variables remain.
3. Merge encoded columns back into the main dataset.

📌 Reminder: Encoding comes after handling missing values and outliers, but before scaling (if applicable).



🌟 Exercise 7: Data Transformation for Age Feature
Goal: Create and encode age groups.

Use pd.cut() to create bins for life stages (e.g. child, teen, adult, senior).
Apply one-hot encoding using pd.get_dummies().
📌 Example: You might define bins like [0, 12, 18, 60, 100] and label them accordingly.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder


## Load Titanic Dataset

In [ ]:

# Load dataset
df = pd.read_csv("Titanic_data/train.csv")

# Display first rows
print(df.head())

# Dataset shape
print("\nDataset Shape:", df.shape)


## Exercise 1: Duplicate Detection and Removal

In [ ]:

# Check duplicates
duplicates = df.duplicated().sum()

print("Number of duplicate rows:", duplicates)

# Shape before removal
before_rows = df.shape[0]

# Remove duplicates
df = df.drop_duplicates()

# Shape after removal
after_rows = df.shape[0]

print("Rows before:", before_rows)
print("Rows after:", after_rows)


## Exercise 2: Handling Missing Values

In [ ]:

# Missing values
print(df.isnull().sum())

# Fill Age with median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill Embarked with mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Fill Cabin with constant value
df["Cabin"] = df["Cabin"].fillna("Unknown")

print("\nMissing values after cleaning:")
print(df.isnull().sum())


## Exercise 3: Feature Engineering

In [ ]:

# Create FamilySize feature
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Extract Title from Name
df["Title"] = df["Name"].str.extract(' ([A-Za-z]+)\.', expand=False)

print(df[["Name", "Title", "FamilySize"]].head())


In [ ]:

# Encode Title using Label Encoding
label_encoder = LabelEncoder()

df["Title_Encoded"] = label_encoder.fit_transform(df["Title"])

print(df[["Title", "Title_Encoded"]].head())


## Exercise 4: Outlier Detection and Handling

In [ ]:

# Boxplot before treatment
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.boxplot(y=df["Fare"])
plt.title("Fare Before Treatment")

plt.subplot(1,2,2)
sns.boxplot(y=df["Age"])
plt.title("Age Before Treatment")

plt.show()


In [ ]:

# IQR Method for Fare

Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap outliers
df["Fare"] = np.where(
    df["Fare"] > upper_bound,
    upper_bound,
    df["Fare"]
)

print("Outliers handled using IQR capping.")


In [ ]:

# Boxplot after treatment
sns.boxplot(y=df["Fare"])
plt.title("Fare After Treatment")
plt.show()


## Exercise 5: Data Standardization and Normalization

In [ ]:

# StandardScaler for Age
standard_scaler = StandardScaler()

df["Age_Standardized"] = standard_scaler.fit_transform(df[["Age"]])

# MinMaxScaler for Fare
minmax_scaler = MinMaxScaler()

df["Fare_Normalized"] = minmax_scaler.fit_transform(df[["Fare"]])

print(df[[
    "Age",
    "Age_Standardized",
    "Fare",
    "Fare_Normalized"
]].head())


## Exercise 6: Feature Encoding

In [ ]:

# One-Hot Encoding
encoded_df = pd.get_dummies(
    df,
    columns=["Sex", "Embarked", "Title"],
    drop_first=True
)

print(encoded_df.head())


## Exercise 7: Data Transformation for Age Feature

In [ ]:

# Create age groups
bins = [0, 12, 18, 60, 100]
labels = ["Child", "Teen", "Adult", "Senior"]

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels
)

print(df[["Age", "AgeGroup"]].head())


In [ ]:

# One-hot encoding for AgeGroup
agegroup_encoded = pd.get_dummies(
    df["AgeGroup"],
    prefix="AgeGroup"
)

print(agegroup_encoded.head())



## Final Insights

- Missing values were successfully handled.
- Duplicate rows were removed.
- New features such as `FamilySize` and `Title` were created.
- Outliers in the `Fare` column were treated using the IQR method.
- Numerical data was standardized and normalized.
- Categorical variables were encoded for machine learning preparation.
- Age groups were created and transformed into numerical features.
